In [ ]:
# import pandas as pd
# import numpy as np

# # -------- Base Strategy Condition -------- #
# class StrategyCondition:
#     def check(self, row) -> bool:
#         raise NotImplementedError("조건 클래스는 반드시 check() 메서드를 구현해야 합니다.")

# # -------- 개별 조건 구현 -------- #
# class RSIBelow(StrategyCondition):
#     def __init__(self, threshold=45):
#         self.threshold = threshold
#     def check(self, row):
#         return row.get("RSI (14일)", 100) < self.threshold

# class BollingerNearLower(StrategyCondition):
#     def __init__(self, buffer_pct=0.02):
#         self.buffer_pct = buffer_pct
#     def check(self, row):
#         return row['종가'] < row.get("볼린저밴드 하단", row['종가']) * (1 + self.buffer_pct)

# class MACDPositive(StrategyCondition):
#     def check(self, row):
#         return row.get("MACD", 0) > row.get("MACD 시그널", 0)

# class MA5AboveMA10(StrategyCondition):
#     def check(self, row):
#         return row.get("SMA 5일", 0) > row.get("SMA 10일", 0)

# class MA5BelowMA60(StrategyCondition):
#     def check(self, row):
#         return row.get("SMA 5일", 0) < row.get("SMA 60일", 0)

# class TwoWeekPriceLow(StrategyCondition):
#     def __init__(self, max_pct=3):
#         self.max_pct = max_pct
#     def check(self, row):
#         return row.get("가격 상승률 (2주)", 100) < self.max_pct

# # -------- Composite Strategy -------- #
# class CompositeStrategy:
#     def __init__(self, conditions: list):
#         self.conditions = conditions
#         self.trigger_dates = []

#     def should_buy(self, row):
#         all_pass = all(cond.check(row) for cond in self.conditions)
#         if all_pass:
#             self.trigger_dates.append(pd.to_datetime(row['날짜']))  # 날짜 저장 (datetime 형식)
#         return all_pass

#     def get_trigger_count(self, start_date, end_date):
#         return len([d for d in self.trigger_dates if start_date <= d < end_date])

# # -------- Backtest Class -------- #
# class BacktestPeriod:
#     def __init__(self, df, start_date, end_date, strategy=None):
#         self.df = df[(df['날짜'] >= start_date) & (df['날짜'] < end_date)].copy()
#         self.cash = 10_000.0
#         self.shares = 0.0
#         self.total_cost = 0.0
#         self.cooldown = 0
#         self.strategy = strategy
#         self.start = pd.to_datetime(start_date)
#         self.end = pd.to_datetime(end_date)

#     def run(self):
#         if self.strategy:
#             self.strategy.trigger_dates = []  # 트리거 날짜 초기화

#         for _, row in self.df.iterrows():
#             price = row['종가']
#             if self.cooldown > 0:
#                 self.cooldown -= 1
#                 continue

#             if self.strategy is None:
#                 if self.shares == 0:
#                     self._buy(price)
#             elif self.strategy.should_buy(row):
#                 self._buy(price)
#         return self._summary()

#     def _buy(self, price):
#         if self.cash > 0:
#             buy_amount = self.cash * 0.1
#             self.shares += buy_amount / price
#             self.cash -= buy_amount
#             self.total_cost += buy_amount
#             self.cooldown = 5

#     def _summary(self):
#         final_value = self.cash + self.shares * self.df.iloc[-1]['종가']
#         roi = (final_value - 10_000) / 10_000 * 100
#         value_to_cost = final_value / self.total_cost if self.total_cost > 0 else np.nan
#         trigger_count = self.strategy.get_trigger_count(self.start, self.end) if self.strategy else "N/A"
#         trigger_note = "조건 미충족" if trigger_count == 0 else f"{trigger_count}회 조건 성립"
#         trigger_dates = self.strategy.trigger_dates if self.strategy else []

#         return {
#             "시작일": self.start.date(),
#             "종료일": self.end.date(),
#             "최종 자산": round(final_value, 2),
#             "투자 비용": round(self.total_cost, 2),
#             "수익률 (%)": round(roi, 2),
#             "자산/매입비용": round(value_to_cost, 2) if not np.isnan(value_to_cost) else "N/A",
#             "매수 조건 성립": trigger_note,
#             "조건 성립 날짜": [d.strftime('%Y-%m-%d') for d in trigger_dates],
#             "전략": "조건 전략" if self.strategy else "베이스라인"
#         }

# # -------- 실행 코드 -------- #
# # CSV 파일 불러오기 (경로는 너의 환경에 맞게 바꿔줘)
# df = pd.read_csv(r"C:\Users\seung\OneDrive\주식\Back Test\Tesla\TSLA_지표포함_이평포함.csv", encoding='utf-8-sig')
# df['날짜'] = pd.to_datetime(df['날짜'])

# # 전략 조건 리스트
# conditions = [
#     RSIBelow(45),
#     BollingerNearLower(0.02),
#     MACDPositive(),
#     MA5AboveMA10(),
#     MA5BelowMA60(),
#     TwoWeekPriceLow(3)
# ]
# composite_strategy = CompositeStrategy(conditions)

# # 시뮬레이션 기간
# periods = [
#     ("2022-01-01", "2023-01-01"),
#     ("2023-01-01", "2024-01-01"),
#     ("2024-01-01", "2025-01-01")
# ]

# # 시뮬레이션 실행
# results = []
# for start, end in periods:
#     results.append(BacktestPeriod(df, start, end, None).run())  # 베이스라인
#     results.append(BacktestPeriod(df, start, end, composite_strategy).run())  # 조건 전략

# # 결과 출력
# results_df = pd.DataFrame(results)
# print(results_df)


# # 결과 CSV 저장 경로 지정
# save_path = r"C:\Users\seung\OneDrive\주식\Back Test\Tesla\TSLA_전략_결과.csv"

# # CSV 파일로 저장
# results_df.to_csv(save_path, index=False, encoding="utf-8-sig")

# print(f"📁 결과 저장 완료 → {save_path}")

In [ ]:
import pandas as pd
import numpy as np

# -------------------- Condition Classes -------------------- #
class StrategyCondition:
    def check(self, row) -> bool:
        raise NotImplementedError()

# --- Buy Conditions --- #
class RSIBelow(StrategyCondition):
    def __init__(self, threshold): self.threshold = threshold
    def check(self, row): return row.get("RSI (14일)", 100) < self.threshold

class BollingerNearLower(StrategyCondition):
    def __init__(self, buffer_pct): self.buffer_pct = buffer_pct
    def check(self, row): return row['종가'] < row.get("볼린저밴드 하단", row['종가']) * (1 + self.buffer_pct)

class MACDPositive(StrategyCondition):
    def check(self, row): return row.get("MACD", 0) > row.get("MACD 시그널", 0)

class MA5AboveMA10(StrategyCondition):
    def check(self, row): return row.get("SMA 5일", 0) > row.get("SMA 10일", 0)

class MA5BelowMA60(StrategyCondition):
    def check(self, row): return row.get("SMA 5일", 0) < row.get("SMA 60일", 0)

class TwoWeekPriceLow(StrategyCondition):
    def __init__(self, max_pct): self.max_pct = max_pct
    def check(self, row): return row.get("가격 상승률 (2주)", 100) < self.max_pct

# --- Sell Conditions --- #
class RSISell(StrategyCondition):
    def __init__(self, threshold): self.threshold = threshold
    def check(self, row): return row.get("RSI (14일)", 0) > self.threshold

class DeathCrossSell(StrategyCondition):
    def check(self, row): return row.get("SMA 5일", 0) < row.get("SMA 20일", 0)

class BollingerUpperSell(StrategyCondition):
    def check(self, row): return row['종가'] > row.get("볼린저밴드 상단", row['종가'])

class VolumeSpikeSell(StrategyCondition):
    def __init__(self, multiple=2): self.multiple = multiple
    def check(self, row):
        try:
            today_vol = float(str(row["거래량"]).replace("M", ""))
            prev_vol = float(str(row.get("전일 거래량", "0")).replace("M", ""))
            prev_prev_vol = float(str(row.get("전전일 거래량", "0")).replace("M", ""))
            if prev_prev_vol == 0: return False
            spike_ratio = prev_vol / prev_prev_vol
            return spike_ratio >= self.multiple and today_vol > prev_vol
        except:
            return False

# -------------------- Strategy Class -------------------- #
class CompositeStrategy:
    def __init__(self, buy_conditions, sell_conditions):
        self.buy_conditions = buy_conditions
        self.sell_conditions = sell_conditions
        self.buy_dates = []
        self.sell_dates = []

    def should_buy(self, row):
        if all(cond.check(row) for cond in self.buy_conditions):
            self.buy_dates.append(pd.to_datetime(row['날짜']))
            return True
        return False

    def should_sell(self, row):
        if any(cond.check(row) for cond in self.sell_conditions):
            self.sell_dates.append(pd.to_datetime(row['날짜']))
            return True
        return False

# -------------------- Backtest Class -------------------- #
class BacktestPeriod:
    def __init__(self, df, start_date, end_date, strategy):
        self.df = df[(df['날짜'] >= start_date) & (df['날짜'] < end_date)].copy()
        self.cash = 10_000.0
        self.shares = 0.0
        self.total_cost = 0.0
        self.cooldown = 0
        self.strategy = strategy
        self.start = pd.to_datetime(start_date)
        self.end = pd.to_datetime(end_date)

    def run(self):
        self.df["전일 거래량"] = self.df["거래량"].shift(1)
        self.df["전전일 거래량"] = self.df["거래량"].shift(2)
        for _, row in self.df.iterrows():
            if self.cooldown > 0: self.cooldown -= 1; continue
            price = row['종가']
            if self.strategy.should_buy(row):
                self._buy(price)
            elif self.strategy.should_sell(row):
                self._sell(price)
        return self._summary()

    def _buy(self, price):
        if self.cash > 0:
            amt = self.cash * 0.1
            self.shares += amt / price
            self.cash -= amt
            self.total_cost += amt
            self.cooldown = 5

    def _sell(self, price):
        if self.shares > 0:
            proceeds = self.shares * price
            self.cash += proceeds
            self.shares = 0
            self.total_cost = 0
            self.cooldown = 5

    def _summary(self):
        final_val = self.cash + self.shares * self.df.iloc[-1]['종가']
        roi = (final_val - 10_000) / 10_000 * 100
        return {
            "시작일": self.start.date(),
            "종료일": self.end.date(),
            "최종 자산": round(final_val, 2),
            "수익률 (%)": round(roi, 2),
            "매수일 수": len(self.strategy.buy_dates),
            "매도일 수": len(self.strategy.sell_dates),
            "매수 날짜": [d.strftime('%Y-%m-%d') for d in self.strategy.buy_dates],
            "매도 날짜": [d.strftime('%Y-%m-%d') for d in self.strategy.sell_dates]
        }


In [ ]:
# --- 시뮬레이션 실행 및 저장 --- #

# (1) 매수 조건
buy_conditions = [
    RSIBelow(45),
    BollingerNearLower(0.02),
    MACDPositive(),
    MA5AboveMA10(),
    MA5BelowMA60(),
    TwoWeekPriceLow(3)
]

# (2) 매도 조건
sell_conditions = [
    RSISell(70),
    DeathCrossSell(),
    BollingerUpperSell(),
    VolumeSpikeSell(multiple=2)
]

# (3) 전략 조합
strategy = CompositeStrategy(buy_conditions, sell_conditions)

# (4) 데이터 로딩 (날짜 + 전일/전전일 거래량 준비)
df = pd.read_csv(r"C:\Users\seung\OneDrive\주식\Back Test\Tesla\TSLA_지표포함_이평포함.csv", encoding="utf-8-sig")
df["날짜"] = pd.to_datetime(df["날짜"])
df["전일 거래량"] = df["거래량"].shift(1)
df["전전일 거래량"] = df["거래량"].shift(2)

# (5) 평가 기간
periods = [
    ("2022-01-01", "2023-01-01"),
    ("2023-01-01", "2024-01-01"),
    ("2024-01-01", "2025-01-01")
]

# (6) 시뮬레이션 반복 실행
results = []
for start, end in periods:
    bt = BacktestPeriod(df, start, end, strategy)
    result = bt.run()
    results.append(result)

# (7) 결과 정리 및 저장
results_df = pd.DataFrame(results)
save_path = r"C:\Users\seung\OneDrive\주식\Back Test\Tesla\TSLA_매수매도_전략결과.csv"
results_df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("✅ 백테스트 완료 및 CSV 저장 완료")
print(results_df)


In [ ]:
# 재시작된 환경에서 필요한 코드 재정의
import pandas as pd
import numpy as np

# ----------- Buy Conditions ----------- #
class StrategyCondition:
    def check(self, row) -> bool:
        raise NotImplementedError()

class RSIBelow(StrategyCondition):
    def __init__(self, threshold): self.threshold = threshold
    def check(self, row): return row.get("RSI (14일)", 100) < self.threshold

class BollingerNearLower(StrategyCondition):
    def __init__(self, buffer_pct): self.buffer_pct = buffer_pct
    def check(self, row): return row['종가'] < row.get("볼린저밴드 하단", row['종가']) * (1 + self.buffer_pct)

class MACDPositive(StrategyCondition):
    def check(self, row): return row.get("MACD", 0) > row.get("MACD 시그널", 0)

class MA5AboveMA10(StrategyCondition):
    def check(self, row): return row.get("SMA 5일", 0) > row.get("SMA 10일", 0)

class MA5BelowMA60(StrategyCondition):
    def check(self, row): return row.get("SMA 5일", 0) < row.get("SMA 60일", 0)

class TwoWeekPriceLow(StrategyCondition):
    def __init__(self, max_pct): self.max_pct = max_pct
    def check(self, row): return row.get("가격 상승률 (2주)", 100) < self.max_pct

# ----------- Sell Conditions ----------- #
class RSISell(StrategyCondition):
    def __init__(self, threshold): self.threshold = threshold
    def check(self, row): return row.get("RSI (14일)", 0) > self.threshold

# ----------- Composite Strategy ----------- #
class CompositeStrategy:
    def __init__(self, buy_conditions, sell_conditions):
        self.buy_conditions = buy_conditions
        self.sell_conditions = sell_conditions
        self.buy_dates = []
        self.sell_dates = []

    def should_buy(self, row):
        if all(cond.check(row) for cond in self.buy_conditions):
            self.buy_dates.append(pd.to_datetime(row['날짜']))
            return True
        return False

    def should_sell(self, row):
        if any(cond.check(row) for cond in self.sell_conditions):
            self.sell_dates.append(pd.to_datetime(row['날짜']))
            return True
        return False

# ----------- Backtest Class ----------- #
class BacktestPeriod:
    def __init__(self, df, start_date, end_date, strategy=None):
        self.df = df[(df['날짜'] >= start_date) & (df['날짜'] < end_date)].copy()
        self.cash = 10_000.0
        self.shares = 0.0
        self.total_cost = 0.0
        self.cooldown = 0
        self.strategy = strategy
        self.start = pd.to_datetime(start_date)
        self.end = pd.to_datetime(end_date)
        self.bought_once = False

    def run(self):
        if self.strategy:
            self.strategy.buy_dates = []
            self.strategy.sell_dates = []

        for _, row in self.df.iterrows():
            price = row['종가']
            if self.cooldown > 0:
                self.cooldown -= 1
                continue

            if self.strategy is None:
                if self.shares == 0:
                    self._buy(price)
            else:
                if self.strategy.should_buy(row):
                    self._buy(price)
                elif self.strategy.should_sell(row) and self.bought_once:
                    self._sell(price)

        return self._summary()

    def _buy(self, price):
        if self.cash > 0:
            buy_amount = self.cash * 0.1
            self.shares += buy_amount / price
            self.cash -= buy_amount
            self.total_cost += buy_amount
            self.cooldown = 5
            self.bought_once = True

    def _sell(self, price):
        if self.shares > 0:
            proceeds = self.shares * price
            self.cash += proceeds
            self.shares = 0
            self.total_cost = 0
            self.cooldown = 5

    def _summary(self):
        final_value = self.cash + self.shares * self.df.iloc[-1]['종가']
        roi = (final_value - 10_000) / 10_000 * 100
        value_to_cost = final_value / self.total_cost if self.total_cost > 0 else np.nan
        return {
            "시작일": self.start.date(),
            "종료일": self.end.date(),
            "최종 자산": round(final_value, 2),
            "투자 비용": round(self.total_cost, 2),
            "수익률 (%)": round(roi, 2),
            "자산/매입비용": round(value_to_cost, 2) if not np.isnan(value_to_cost) else "N/A",
            "매수일 수": len(self.strategy.buy_dates) if self.strategy else "N/A",
            "매도일 수": len(self.strategy.sell_dates) if self.strategy else "N/A",
            "매수 날짜": [d.strftime('%Y-%m-%d') for d in self.strategy.buy_dates] if self.strategy else [],
            "매도 날짜": [d.strftime('%Y-%m-%d') for d in self.strategy.sell_dates] if self.strategy else [],
            "전략": "조건 전략" if self.strategy else "베이스라인"
        }


In [ ]:
# 조건 설정
buy_conditions = [RSIBelow(45), BollingerNearLower(0.02), MACDPositive(), MA5AboveMA10(), MA5BelowMA60(), TwoWeekPriceLow(3)]
sell_conditions = [RSISell(70)]
strategy = CompositeStrategy(buy_conditions, sell_conditions)

# 시뮬레이션 실행
backtest = BacktestPeriod(df, "2023-01-01", "2024-01-01", strategy)
print(backtest.run())
baseline = BacktestPeriod(df, "2023-01-01", "2024-01-01", None)
print(baseline.run())



In [ ]:
# --- 시뮬레이션 실행 및 저장 --- #

# (1) 매수 조건
buy_conditions = [
    RSIBelow(45),
    BollingerNearLower(0.02),
    MACDPositive(),
    MA5AboveMA10(),
    MA5BelowMA60(),
    TwoWeekPriceLow(3)
]

# (2) 매도 조건
sell_conditions = [
    RSISell(70),
    DeathCrossSell(),
    BollingerUpperSell(),
    VolumeSpikeSell(multiple=2)
]

# (3) 전략 조합
strategy = CompositeStrategy(buy_conditions, sell_conditions)


# (4) 데이터 로딩 (날짜 + 전일/전전일 거래량 준비)
df = pd.read_csv(r"C:\Users\seung\OneDrive\주식\Back Test\Tesla\TSLA_지표포함_이평포함.csv", encoding="utf-8-sig")
df["날짜"] = pd.to_datetime(df["날짜"])
df["전일 거래량"] = df["거래량"].shift(1)
df["전전일 거래량"] = df["거래량"].shift(2)

# (5) 평가 기간
periods = [
    ("2022-01-01", "2023-01-01"),
    ("2023-01-01", "2024-01-01"),
    ("2024-01-01", "2025-01-01")
]

# (6) 시뮬레이션 반복 실행
results = []
for start, end in periods:
    bt = BacktestPeriod(df, start, end, strategy)
    result = bt.run()
    results.append(result)

# (7) 결과 정리 및 저장
results_df = pd.DataFrame(results)
save_path = r"C:\Users\seung\OneDrive\주식\Back Test\Tesla\TSLA_매수매도_전략결과.csv"
results_df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("✅ 백테스트 완료 및 CSV 저장 완료")
print(results_df)


In [ ]:
########

In [ ]:
# 이전에 작성된 BacktestPeriodModified 클래스에
# 매수 타이밍 가격 및 매도 타이밍 가격 기록 기능 추가

class BacktestPeriodModified:
    def __init__(self, df, start_date, end_date, strategy=None, allow_sell=True):
        self.df = df[(df['날짜'] >= start_date) & (df['날짜'] < end_date)].copy()
        self.cash = 10_000.0
        self.shares = 0.0
        self.total_cost = 0.0
        self.cooldown = 0
        self.strategy = strategy
        self.start = pd.to_datetime(start_date)
        self.end = pd.to_datetime(end_date)
        self.allow_sell = allow_sell
        self.buy_lots = []  # (날짜, 가격, 수량)
        self.bought_once = False
        self.buy_prices = []  # 매수 가격 기록
        self.sell_prices = []  # 매도 가격 기록

    def run(self):
        if self.strategy:
            self.strategy.buy_dates = []
            self.strategy.sell_dates = []

        for _, row in self.df.iterrows():
            price = row['종가']
            if self.cooldown > 0:
                self.cooldown -= 1
                continue

            if self.strategy is None:
                if self.shares == 0:
                    self._buy(price, row['날짜'])
            else:
                if self.strategy.should_buy(row):
                    self._buy(price, row['날짜'])
                elif self.allow_sell and self.strategy.should_sell(row) and self.shares > 0:
                    self._sell(price, row['날짜'])

        return self._summary()

    def _buy(self, price, date):
        if self.cash > 0:
            buy_amount = self.cash * 0.1
            num_shares = buy_amount / price
            self.shares += num_shares
            self.cash -= buy_amount
            self.total_cost += buy_amount
            self.buy_lots.append((date, price, num_shares))
            self.cooldown = 5
            self.bought_once = True
            self.buy_prices.append((date, round(price, 2)))

    def _sell(self, price, date):
        if self.shares > 0 and len(self.buy_lots) > 0:
            proceeds = self.shares * price
            self.cash += proceeds
            self.shares = 0
            self.total_cost = 0
            self.buy_lots.clear()
            self.cooldown = 5
            self.sell_prices.append((date, round(price, 2)))
            if self.strategy:
                self.strategy.sell_dates.append(pd.to_datetime(date))

    def _summary(self):
        final_value = self.cash + self.shares * self.df.iloc[-1]['종가']
        roi = (final_value - 10_000) / 10_000 * 100
        value_to_cost = final_value / self.total_cost if self.total_cost > 0 else np.nan
        return {
            "시작일": self.start.date(),
            "종료일": self.end.date(),
            "최종 자산": round(final_value, 2),
            "투자 비용": round(self.total_cost, 2),
            "수익률 (%)": round(roi, 2),
            "자산/매입비용": round(value_to_cost, 2) if not np.isnan(value_to_cost) else "N/A",
            "매수일 수": len(self.strategy.buy_dates) if self.strategy else "N/A",
            "매도일 수": len(self.strategy.sell_dates) if self.strategy else "N/A",
            "매수 날짜": [d.strftime('%Y-%m-%d') for d in self.strategy.buy_dates] if self.strategy else [],
            "매수 가격": [f"{d.strftime('%Y-%m-%d')}: ${p}" for d, p in self.buy_prices],
            "매도 날짜": [d.strftime('%Y-%m-%d') for d in self.strategy.sell_dates] if self.strategy else [],
            "매도 가격": [f"{d.strftime('%Y-%m-%d')}: ${p}" for d, p in self.sell_prices],
            "전략": "조건 전략" if self.strategy else "베이스라인" + (" (매도X)" if not self.allow_sell else "")
        }


In [ ]:
# 데이터 로드
df = pd.read_csv(r"C:\Users\seung\OneDrive\주식\Back Test\Tesla\TSLA_지표포함_이평포함.csv", encoding="utf-8-sig")
df["날짜"] = pd.to_datetime(df["날짜"])

# 전략 조건 정의
buy_conditions = [
    RSIBelow(45),
    BollingerNearLower(0.02),
    MACDPositive(),
    MA5AboveMA10(),
    MA5BelowMA60(),
    TwoWeekPriceLow(3)
]
sell_conditions = [RSISell(70)]
strategy = CompositeStrategy(buy_conditions, sell_conditions)

# 평가 기간
periods = [
    ("2022-01-01", "2023-01-01"),
    ("2023-01-01", "2024-01-01"),
    ("2024-01-01", "2025-01-01")
]

# 백테스트 실행
results = []
for start, end in periods:
    results.append(BacktestPeriodModified(df, start, end, None).run())  # 베이스라인
    results.append(BacktestPeriodModified(df, start, end, strategy, allow_sell=True).run())  # 조건 전략
    results.append(BacktestPeriodModified(df, start, end, strategy, allow_sell=False).run())  # 조건 전략(매도X)

# 저장
results_df = pd.DataFrame(results)
save_path = r"C:\Users\seung\OneDrive\주식\Back Test\Tesla\TSLA_전략_결과.csv"
results_df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("📁 저장 완료:", save_path)
results_df
